# Stat 220, Unit 1 Homework: Statistical Inference

Answer each problem in the cell(s) provided. Replace *Your answer* with your response
for written parts, and put code in the empty code cells.

The first problem is a **simulation lab**: you build a world where you already know the truth, so
you can watch the machinery work. The next three are **real data with real decisions attached**.
The last one asks what you are and are not entitled to conclude.

Throughout, a correct number with a wrong interpretation earns very little. Say what the number
means, and say what it does not mean.

**Data** (real, all from the course server):

- `https://richardson.byu.edu/220/bikes.csv`: 365 days of bike-rental counts with weather.
- `https://richardson.byu.edu/220/rent.csv`: 4,743 rental listings across six cities.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(220)
bikes = pd.read_csv("https://richardson.byu.edu/220/bikes.csv")
rent = pd.read_csv("https://richardson.byu.edu/220/rent.csv")
print(bikes.shape, rent.shape)
bikes.head()

**Problem 1.** *Simulation lab: what a p-value is.* Here you control the truth, so you can check
what the machinery does. Use `rng` for everything.

Part a. Write a function `run_studies(n_studies, n, true_diff, sd=1.0)` that simulates
`n_studies` experiments. Each experiment draws two independent groups of size `n` from normal
distributions whose means differ by `true_diff`, runs a Welch two-sample $t$-test, and returns the
array of p-values.

In [ ]:
def run_studies(n_studies, n, true_diff, sd=1.0):
    p_values = []

    for _ in range(n_studies):
        group1 = rng.normal(loc=0, scale=sd, size=n)
        group2 = rng.normal(loc=true_diff, scale=sd, size=n)

        result = stats.ttest_ind(group1, group2, equal_var=False)

        p_values.append(result.pvalue)

    return np.array(p_values)

Part b. Run 5,000 studies with `true_diff = 0` and $n = 30$, so that **nothing is really
going on**. Plot a histogram of the p-values, and report the fraction below 0.05.

In [ ]:
p_values = run_studies(
    n_studies=5000,
    n=30,
    true_diff=0
)

plt.hist(p_values, bins=20, edgecolor="black")
plt.xlabel("p-value")
plt.ylabel("Number of studies")
plt.title("P-values when the null hypothesis is true")
plt.show()

fraction = np.mean(p_values < 0.05)
print(f"Fraction with p < 0.05: {fraction:.3f}")

Part c. The histogram should look flat. Explain in 2 to 3 sentences why the p-values are
uniform when the null is true, and what that implies about the meaning of "$p < 0.05$".

When the null hypothesis is true, p-values are approximately uniform because every value between 0 and 1 is equally plausible under repeated random sampling. Therefore, \(p<0.05\) means that the observed result is among the most unusual 5% of results expected when the null hypothesis is true; it does not mean that there is a 5% chance that the null hypothesis itself is true.



Part d. Now set `true_diff = 0.5` and compute the fraction of studies that reach $p<0.05$
for $n = 10, 30, 100, 400$. This fraction is the **power**. Present the four numbers in a small
table and describe the pattern.

In [ ]:
sample_sizes = [10, 30, 100, 400]

power_results = []

for n in sample_sizes:
    p_values = run_studies(
        n_studies=5000,
        n=n,
        true_diff=0.5
    )

    power = np.mean(p_values < 0.05)
    power_results.append(power)

power_table = pd.DataFrame({
    "n": sample_sizes,
    "power": power_results
})

print(power_table)

Part e. Keep $n = 15$ and `true_diff = 0.3`. Among only the studies that reached
significance, compute the average estimated difference. Compare it to the true 0.3 and explain the
phenomenon you have just measured, and why it makes small significant studies untrustworthy.

In [ ]:
n_studies = 5000

estimated_diffs = []
p_values = []

for _ in range(n_studies):
    group1 = rng.normal(0, 1, 15)
    group2 = rng.normal(0.3, 1, 15)

    result = stats.ttest_ind(group1, group2, equal_var=False)

    estimated_diffs.append(group2.mean() - group1.mean())
    p_values.append(result.pvalue)

estimated_diffs = np.array(estimated_diffs)
p_values = np.array(p_values)

significant = p_values < 0.05

print("Number significant:", significant.sum())
print("Average estimated difference among significant studies:",
      estimated_diffs[significant].mean())
print("True difference:", 0.3)

Part f. Simulate the peeking problem: run experiments with **no real effect** where you
test after every 20 new observations up to $n = 500$ and stop the moment $p < 0.05$. Across 2,000
such experiments, what fraction produce a "significant" result? Explain the gap between that
number and 0.05.

In [ ]:
n_experiments = 2000
results = []

for _ in range(n_experiments):
    group1 = []
    group2 = []
    significant = False

    for n in range(20, 501, 20):
        group1.extend(rng.normal(0, 1, 20))
        group2.extend(rng.normal(0, 1, 20))

        result = stats.ttest_ind(
            group1,
            group2,
            equal_var=False
        )

        if result.pvalue < 0.05:
            significant = True
            break

    results.append(significant)

fraction_significant = np.mean(results)
print(f"Fraction significant: {fraction_significant:.3f}")

**Problem 2.** *Real data: do holidays change bike ridership?* Use `bikes`. The operations team
wants to know whether to staff differently on holidays. `Count` is the day's rentals and `Holiday`
marks the day type.

Part a. Report the number of days, mean, and standard deviation of `Count` for each level
of `Holiday`. Plot the two distributions side by side (boxplot or overlaid histograms).

In [ ]:
bikes.groupby("Holiday")["Count"].agg(["count", "mean", "std"])

bikes.boxplot(
    column="Count",
    by="Holiday"
)

plt.title("Bike rentals by holiday status")
plt.suptitle("")
plt.xlabel("Holiday")
plt.ylabel("Daily rentals")
plt.show()

Part b. Run a Welch two-sample $t$-test comparing the two groups. Report the difference in
means, the $t$-statistic, the p-value, and a 95% confidence interval for the difference.

In [ ]:
holiday = bikes.loc[bikes["Holiday"] == "Holiday", "Count"]
nonholiday = bikes.loc[bikes["Holiday"] == "No Holiday", "Count"]

result = stats.ttest_ind(
    holiday,
    nonholiday,
    equal_var=False
)

print("Difference:", holiday.mean() - nonholiday.mean())
print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)

Part c. The result is not significant. Your manager writes: *"Confirmed, holidays make no difference. Staff them normally."* Using your confidence interval, explain in 3 to 4 sentences
why that conclusion is not supported, and what the data actually establishes.

The confidence interval for the difference is approximately −377 to 213 rentals per day, so the data are compatible with holidays producing either somewhat fewer or somewhat more rentals. Because the interval includes zero, this study does not provide statistically significant evidence of a difference in mean ridership. However, failing to find a significant difference does not establish that holidays make no difference. The data establish that this one-year sample does not estimate the holiday effect precisely enough to rule out differences of practical size.

Part d. Quantify the problem: by simulation, estimate the power this comparison would have
to detect a true difference of 150 rentals per day, given the group sizes and standard deviations
you observed. (Simulate from normals with those parameters and count how often $p<0.05$.) What
does the number tell you?

In [ ]:
n_holiday = len(holiday)
n_nonholiday = len(nonholiday)

sd_holiday = holiday.std()
sd_nonholiday = nonholiday.std()

n_sim = 5000
p_values = []

for _ in range(n_sim):
    simulated_holiday = rng.normal(
        loc=150,
        scale=sd_holiday,
        size=n_holiday
    )

    simulated_nonholiday = rng.normal(
        loc=0,
        scale=sd_nonholiday,
        size=n_nonholiday
    )

    result = stats.ttest_ind(
        simulated_holiday,
        simulated_nonholiday,
        equal_var=False
    )

    p_values.append(result.pvalue)

power = np.mean(np.array(p_values) < 0.05)
print("Estimated power:", power)

Part e. How many holidays would you need in the data to reach 80% power against a
150-rental difference? Find it by simulating a range of holiday counts. Given that a year contains
a fixed number of holidays, what does your answer imply about ever settling this question with one
year of data?

In [ ]:
n_nonholiday = len(nonholiday)
sd_holiday = holiday.std()
sd_nonholiday = nonholiday.std()

holiday_counts = range(20, 201, 10)
powers = []

for n_holiday in holiday_counts:
    p_values = []

    for _ in range(5000):
        simulated_holiday = rng.normal(
            loc=150,
            scale=sd_holiday,
            size=n_holiday
        )

        simulated_nonholiday = rng.normal(
            loc=0,
            scale=sd_nonholiday,
            size=n_nonholiday
        )

        test = stats.ttest_ind(
            simulated_holiday,
            simulated_nonholiday,
            equal_var=False
        )

        p_values.append(test.pvalue)

    powers.append(np.mean(np.array(p_values) < 0.05))

power_table = pd.DataFrame({
    "Holiday observations": holiday_counts,
    "Power": powers
})

print(power_table)

needed = next(
    n for n, power in zip(holiday_counts, powers)
    if power >= 0.80
)

print("\nNumber needed for 80% power:", needed)

print(
    "A single year has only 18 holidays, so one year of data cannot provide "
    "80% power to detect a 150-rental difference."
)

**Problem 3.** *Real data: furnished apartments rent for more.* Use `rent`. A property manager
asks how much extra to charge for furnishing a unit.

Part a. Compare `Rent` for `FurnishingStatus == "Furnished"` versus `"Unfurnished"`.
Report both sample sizes, both means, the difference, the $t$-statistic, and the p-value.

In [ ]:
furnished = rent.loc[
    rent["FurnishingStatus"] == "Furnished",
    "Rent"
]

unfurnished = rent.loc[
    rent["FurnishingStatus"] == "Unfurnished",
    "Rent"
]

result = stats.ttest_ind(
    furnished,
    unfurnished,
    equal_var=False
)

print("Furnished n:", len(furnished))
print("Unfurnished n:", len(unfurnished))
print("Furnished mean:", furnished.mean())
print("Unfurnished mean:", unfurnished.mean())
print("Difference:", furnished.mean() - unfurnished.mean())
print("t:", result.statistic)
print("p:", result.pvalue)

Part b. The p-value is astronomically small. Explain what that does and does not tell you.
In particular, state whether it says anything about **how much** furnishing is worth.

The extremely small p-value provides very strong evidence that the mean rents differ between furnished and unfurnished listings in this dataset. However, the p-value does not tell us how large or economically important the difference is. In particular, it does not say that furnishing itself is worth $33,691, because the furnished and unfurnished listings may differ in other characteristics that also affect rent.

Part c. Report a 95% confidence interval for the difference in mean rent, and compare the
information it carries to the p-value from Part a. Which would you put in a memo, and why?

In [ ]:
furnished = rent.loc[
    rent["FurnishingStatus"] == "Furnished", "Rent"
]

unfurnished = rent.loc[
    rent["FurnishingStatus"] == "Unfurnished", "Rent"
]

mean_diff = furnished.mean() - unfurnished.mean()

se = np.sqrt(
    furnished.var(ddof=1) / len(furnished)
    + unfurnished.var(ddof=1) / len(unfurnished)
)

df = (
    (furnished.var(ddof=1) / len(furnished)
     + unfurnished.var(ddof=1) / len(unfurnished)) ** 2
    /
    (
        (furnished.var(ddof=1) / len(furnished)) ** 2
        / (len(furnished) - 1)
        +
        (unfurnished.var(ddof=1) / len(unfurnished)) ** 2
        / (len(unfurnished) - 1)
    )
)

margin = stats.t.ppf(0.975, df) * se

lower = mean_diff - margin
upper = mean_diff + margin

print("Difference in means:", mean_diff)
print("95% CI:", (lower, upper))

print(
    "The confidence interval is more useful for a memo because it shows "
    "the plausible size of the rent difference, while the p-value mainly "
    "shows evidence that the means differ."
)

Part d. Furnished units are not otherwise identical to unfurnished ones. Compare the two
groups on `Size`, `BHK`, and `City` composition. Report at least two ways they differ.

In [ ]:
comparison = rent[
    rent["FurnishingStatus"].isin(["Furnished", "Unfurnished"])
].groupby("FurnishingStatus")[["Size", "BHK"]].mean()

print(comparison)

# Compare City composition
city_composition = pd.crosstab(
    rent["City"],
    rent["FurnishingStatus"],
    normalize="columns"
)

print("\nCity composition:")
print(city_composition)

print(
    "Furnished listings are larger and have more bedrooms on average, "
    "and their city composition also differs from unfurnished listings."
)

Part e. Repeat the furnished-versus-unfurnished comparison **within a single city and a
single `BHK` level** (choose ones with enough listings). Report the difference and its interval.
How does it compare to the raw difference from Part a, and what does that tell the property
manager about the number they should actually use?

In [ ]:
subset = rent[
    (rent["City"] == "Mumbai") &
    (rent["BHK"] == 2) &
    (rent["FurnishingStatus"].isin(["Furnished", "Unfurnished"]))
]

furnished_mumbai = subset.loc[
    subset["FurnishingStatus"] == "Furnished", "Rent"
]

unfurnished_mumbai = subset.loc[
    subset["FurnishingStatus"] == "Unfurnished", "Rent"
]

test = stats.ttest_ind(
    furnished_mumbai,
    unfurnished_mumbai,
    equal_var=False
)

difference = furnished_mumbai.mean() - unfurnished_mumbai.mean()

se = np.sqrt(
    furnished_mumbai.var(ddof=1) / len(furnished_mumbai)
    + unfurnished_mumbai.var(ddof=1) / len(unfurnished_mumbai)
)

df = (
    (furnished_mumbai.var(ddof=1) / len(furnished_mumbai)
     + unfurnished_mumbai.var(ddof=1) / len(unfurnished_mumbai)) ** 2
    /
    (
        (furnished_mumbai.var(ddof=1) / len(furnished_mumbai)) ** 2
        / (len(furnished_mumbai) - 1)
        +
        (unfurnished_mumbai.var(ddof=1) / len(unfurnished_mumbai)) ** 2
        / (len(unfurnished_mumbai) - 1)
    )
)

margin = stats.t.ppf(0.975, df) * se

lower = difference - margin
upper = difference + margin

print("City: Mumbai")
print("BHK: 2")
print("Furnished n:", len(furnished_mumbai))
print("Unfurnished n:", len(unfurnished_mumbai))
print("Difference:", difference)
print("t-statistic:", test.statistic)
print("p-value:", test.pvalue)
print("95% CI:", (lower, upper))

print(
    "The within-Mumbai 2-BHK difference is smaller than the raw difference, "
    "so the manager should use a comparison matched to the type of unit being priced."
)

**Problem 4.** *When the test is quietly wrong.* Both datasets violate a $t$-test assumption.
Find the damage.

Part a. Plot `Rent` and describe its shape. Then compare the mean and median. Which
assumption of the $t$-test is under strain here, and does the sample size rescue it?

In [ ]:
plt.hist(rent["Rent"], bins=50, edgecolor="black")
plt.xlabel("Rent")
plt.ylabel("Count")
plt.title("Distribution of Rent")
plt.show()

print("Mean:", rent["Rent"].mean())
print("Median:", rent["Rent"].median())

print(
    "Rent is strongly right-skewed, so the normality assumption is under strain, "
    "although the large sample helps make inference about the mean more robust."
)

Part b. Redo the Problem 3a comparison three ways, all of them $t$-tests: on the raw
`Rent`, on $\log(\text{Rent})$, and on the raw rents after dropping the most extreme 1% of
listings. Report all three and explain what each one is actually estimating, and why they do not
answer quite the same question. (We get a fourth and better way, the bootstrap, in Unit 9.)

In [ ]:
furnished = rent.loc[
    rent["FurnishingStatus"] == "Furnished", "Rent"
]

unfurnished = rent.loc[
    rent["FurnishingStatus"] == "Unfurnished", "Rent"
]

# 1. Raw Rent
raw_test = stats.ttest_ind(
    furnished,
    unfurnished,
    equal_var=False
)

# 2. Log(Rent)
log_furnished = np.log(furnished)
log_unfurnished = np.log(unfurnished)

log_test = stats.ttest_ind(
    log_furnished,
    log_unfurnished,
    equal_var=False
)

# 3. Raw Rent after dropping the highest 1%
cutoff = rent["Rent"].quantile(0.99)

trimmed = rent[
    (rent["Rent"] <= cutoff) &
    (rent["FurnishingStatus"].isin(["Furnished", "Unfurnished"]))
]

trimmed_furnished = trimmed.loc[
    trimmed["FurnishingStatus"] == "Furnished", "Rent"
]

trimmed_unfurnished = trimmed.loc[
    trimmed["FurnishingStatus"] == "Unfurnished", "Rent"
]

trimmed_test = stats.ttest_ind(
    trimmed_furnished,
    trimmed_unfurnished,
    equal_var=False
)

print("Raw Rent:")
print("t =", raw_test.statistic)
print("p =", raw_test.pvalue)

print("\nLog(Rent):")
print("t =", log_test.statistic)
print("p =", log_test.pvalue)

print("\nRaw Rent, highest 1% removed:")
print("Cutoff =", cutoff)
print("t =", trimmed_test.statistic)
print("p =", trimmed_test.pvalue)

print(
    "The three tests differ because raw rent estimates a dollar mean difference, "
    "log rent emphasizes relative differences, and trimming estimates the dollar "
    "difference after excluding the most extreme high-rent listings."
)

Part c. Now the `bikes` data. Compute the lag-1 autocorrelation of `Count` across the
year (`bikes["Count"].autocorr(1)`). Report it and explain what it means about the independence
assumption behind Problem 2.

In [ ]:
autocorrelation = bikes["Count"].autocorr(1)

print("Lag-1 autocorrelation:", autocorrelation)

print(
    "The positive autocorrelation means nearby days have related rental counts, "
    "so the daily observations are not fully independent as assumed in Problem 2."
)

Part d. Demonstrate the damage by simulation: generate two groups of 200 "days" each with
**no real difference** but with each day correlated with the previous one (for example
$x_t = 0.5x_{t-1} + \varepsilon_t$). Across 2,000 such experiments, what fraction produce
$p < 0.05$? Compare it to the 5% you asked for.

In [ ]:
n_experiments = 2000
n_days = 200
p_values = []

for _ in range(n_experiments):

    # Group 1
    x = np.zeros(n_days)
    x[0] = rng.normal()

    for t in range(1, n_days):
        x[t] = 0.5 * x[t - 1] + rng.normal()

    # Group 2
    y = np.zeros(n_days)
    y[0] = rng.normal()

    for t in range(1, n_days):
        y[t] = 0.5 * y[t - 1] + rng.normal()

    test = stats.ttest_ind(
        x,
        y,
        equal_var=False
    )

    p_values.append(test.pvalue)

fraction_significant = np.mean(np.array(p_values) < 0.05)

print("Fraction with p < 0.05:", fraction_significant)
print("Expected fraction if the test were behaving correctly:", 0.05)

print(
    "The correlated data produce significant results much more often than 5%, "
    "showing that ignoring dependence can greatly inflate the false-positive rate."
)

Part e. In 2 to 3 sentences, state which of the two problems, skew or dependence, you would worry about more in a real analysis, and why.

I would worry more about dependence because the bike data show substantial serial correlation, and ignoring it can greatly inflate the false-positive rate, as the simulation demonstrated. Skew can often be mitigated by a large sample size or a transformation, whereas treating strongly dependent observations as independent can make the reported standard errors and p-values seriously misleading.

**Problem 5.** *What can and cannot be said.* No new computation is required here. This is the part that
separates an analyst from a calculator.

Part a. Write the two-sentence result statement you would send the operations team about
holidays and bike ridership (Problem 2). It must be honest about both what you found and what
remains unknown.

In the one-year sample, holiday days averaged about 82 fewer rentals than non-holidays, but the 95% confidence interval ranges from about 377 fewer to 213 more rentals and the difference was not statistically significant. Therefore, the data do not establish that holidays have no effect; substantially more data would be needed to determine whether a practically important difference exists.

Part b. Write the two-sentence statement you would send the property manager about
furnishing (Problem 3), including the number you would recommend and the caveat attached to it.

Furnished listings had an overall mean rent about $33,691 higher than unfurnished listings, but within Mumbai 2-BHK listings the estimated difference was about $25,082, with a 95% confidence interval of approximately $14,231 to $35,933. I would use the matched Mumbai 2-BHK estimate as a more relevant pricing benchmark, while emphasizing that the observational data do not establish that furnishing itself causes the entire difference.

Part c. For each of the following claims, say whether your analyses support it, and if
not, what evidence would be required. (i) *Holidays do not affect ridership.* (ii) *Furnishing a
unit causes rent to rise by the amount you found in Problem 3a.* (iii) *The furnishing result is
more trustworthy than the holiday result because its p-value is smaller.*

**(i) Holidays do not affect ridership — Not supported.** The non-significant test does not prove that the holiday effect is zero; evidence from more years of data or a higher-powered study would be needed to rule out practically important effects.

**(ii) Furnishing a unit causes rent to rise by the amount found in Problem 3a — Not supported.** Problem 3a establishes an association between furnishing status and rent, not causation. Evidence would require controlling for important confounders or, ideally, a randomized experiment in which comparable units are assigned to furnished and unfurnished conditions.

**(iii) The furnishing result is more trustworthy than the holiday result because its p-value is smaller — Not supported.** A smaller p-value does not by itself establish that an analysis is more trustworthy. Trustworthiness depends on the study design, assumptions, dependence, confounding, sample size, and relevance of the comparison to the decision.

Part d. A colleague proposes settling the holiday question by testing every weather
variable, day type, and season in the dataset and reporting whichever comparisons come out
significant. Explain what is wrong with this, and estimate roughly how many "significant" results
they would find by chance if they ran 40 such tests on pure noise.

Testing many variables and reporting whichever ones are significant creates a multiple-testing problem and amounts to searching for significance rather than testing a pre-specified hypothesis. Even if all 40 tests were pure noise, about \(40(0.05)=2\) significant results would be expected by chance on average. The probability of getting at least one false significant result is \(1-0.95^{40}\approx0.871\), or about 87%.

Part e. Across this assignment you used p-values, confidence intervals, effect sizes, and
power. Rank them by how useful they were for making an actual decision, and defend your
ranking in 3 to 4 sentences.

My ranking for decision-making usefulness is **effect size first, confidence interval second, power third, and p-value fourth**. The effect size tells the decision-maker how large the estimated difference is, while the confidence interval shows both its magnitude and uncertainty. Power tells us how capable the study was of detecting an effect of practical interest, which is especially important when interpreting a non-significant result. The p-value is useful for assessing evidence against a null hypothesis, but by itself it does not tell us the size, practical importance, or uncertainty of the effect.